In [ ]:
import pandas as pd
import numpy as np

movies = pd.read_csv("../data/tmdb_5000_movies.csv")
credits = pd.read_csv("../data/tmdb_5000_credits.csv")
movies.head()



## Merge Movie Dataset with Credits Dataset
We merge the `movies` dataset with the `credits` dataset using the movie title.

In [ ]:
movies = movies.merge(credits,on = "title")
movies.head()

## Select Important Columns
We only keep the features needed for recommendation.


In [ ]:
movies = movies[["movie_id","title","overview","genres","keywords","cast","crew"]]
movies.head()

## Check Missing Values
all the missing values in each colums is show below 

In [ ]:
movies.isnull().sum()

## Remove The Missing Values 
we remove the missing values from the data set

In [ ]:
movies.dropna(inplace = True)

## Inspect The Data
Look at one row to understand the structure

In [ ]:
movies.iloc[0]

## Convert String to List
This is actually a list stored as a string, so we must convert it. we have to conver it using library

In [ ]:
import ast

def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i["name"])
    return L


movies['genres'] = movies['genres'].apply(convert)
movies.head()

In [ ]:
movies["keywords"] = movies["keywords"].apply(convert)
movies.head()

## Extract Top 3 Cast Members
we extract cast members from the cast colums

In [ ]:
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter !=3:
            L.append(i['name'])
            counter+=1
        else:
            break
    return L

movies['cast'] = movies['cast'].apply(convert3)
            

## Extract Director from Crew
extracting director from the crew colums

In [ ]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L
    
movies['crew'] = movies['crew'].apply(fetch_director)
movies.head()

## Convert Overview into Words
Overview is a sentence, so split it.

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies.head()

## Remove Spaces From Names
remove space from the names so the model can treat them as one word 

In [ ]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

## Combine All Features Into tags
add the features like 'overview','genres','keywords','cast','crew' in tags

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies.head()

## New DataFrame 
We only need movie_id, title, tags

In [ ]:
new_df = movies[['movie_id','title','tags']]
new_df.head()

## Convert List--> String and Convert to Lowercase 


In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())

In [ ]:
new_df.head()

## Convert Text to Vectors
we will use scikit-learn

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000,stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
vectors.shape

## Reduce Word Variations (Stemming)

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

new_df['tags'] = new_df['tags'].apply(stem)
new_df.head()

## Compute Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)
similarity.shape

## Create Recommendation Function

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    
    for i in movies_list:
        print(new_df.iloc[i[0]].title)
recommend("Batman Begins")

## Save The Model

In [ ]:
import pickle
pickle.dump(new_df, open('movies.pkl','wb'))
pickle.dump(similarity, open('similarity.pkl','wb'))